# Prueba del modelo sobre fotos de dron

Corre el detector sobre las fotos de `Fotos de prueba/` **a la escala correcta**, como las vería la cámara OV9281: la foto se lleva a 5 mm/px, a gris y con ruido de sensor, se corta en tiles de 640×400 (los del entrenamiento) y cada tile entra al modelo sin achicarse.

**Cómo usarlo**

1. Copiar el `best.pt` de `entregables.zip` (salida de `train-model-v3.ipynb` en Kaggle) a `Modelo/Modelo nuevo/best.pt`.
2. Abrir este notebook desde la carpeta `Proyecto final` (Jupyter o VS Code) y ejecutar todo.
3. Los resultados quedan en `Resultados prueba/` (imagen con cajas + tabla de detecciones).

En la PC se usa `best.pt`: el `*_edgetpu.tflite` solo corre en la Coral (se prueba allá con `Coral/bench_fps.py`).


In [ ]:
# ============================================================
# 1. Configuración
# ============================================================
from pathlib import Path
import time
import cv2
import numpy as np
import matplotlib.pyplot as plt
from ultralytics import YOLO

CARPETA = Path.cwd()                      # carpeta "Proyecto final" (donde está este notebook)
FOTOS = CARPETA / "Fotos de prueba"
SALIDA = CARPETA / "Resultados prueba"

RUTA_MODELO = CARPETA / "Modelo" / "Modelo nuevo" / "best.pt"   # best.pt de entregables.zip (train-model-v3)

CONF = 0.25                               # umbral de confianza (usar el sugerido en metricas.json)
SOLAPE = 0.15                             # solape entre tiles
GSD_OBJETIVO_MM = 5.0                     # mm/px de la cámara simulada (igual que en el entrenamiento)

# GSD de cada foto en mm/px. Estimaciones a ojo (Zenmuse X5S, f = 15 mm): corregir si se conoce la distancia.
GSD_FOTO_MM = {
    "DJI_0746": 1.5, "DJI_0748": 1.5,     # detalle: la pala llena el cuadro
    "DJI_0592": 2.5, "DJI_0769": 3.0,     # torre y pala a media distancia
    "DJI_0005": 6.0, "DJI_0044": 6.0,     # vista general de la turbina
}
GSD_POR_DEFECTO = 1.5

TILE = (640, 400)                         # (ancho, alto) de los tiles de entrenamiento

if not RUTA_MODELO.exists():
    raise FileNotFoundError(f"No encuentro {RUTA_MODELO}. Copiar ahí el best.pt de entregables.zip.")
modelo = YOLO(str(RUTA_MODELO), task="detect")
NOMBRES = modelo.names
print(f"Modelo: {RUTA_MODELO.relative_to(CARPETA)}  clases: {NOMBRES}")
print(f"Tile: {TILE}  conf: {CONF}  escala: {GSD_OBJETIVO_MM:g} mm/px")


In [ ]:
# ============================================================
# 2. Funciones
# ============================================================
def leer(p):
    """cv2.imread no lee rutas con acentos en Windows; esto sí."""
    return cv2.imdecode(np.fromfile(str(p), dtype=np.uint8), cv2.IMREAD_COLOR)

def guardar(p, img):
    ok, buf = cv2.imencode(Path(p).suffix, img)
    buf.tofile(str(p))

def degradar(f, rng, fuerte=False):
    """Misma simulación de lente + sensor que en train-model-v3 (versión suave, la de val/test)."""
    h, w = f.shape[:2]
    f = cv2.GaussianBlur(f, (0, 0), rng.uniform(0.5, 1.2) if fuerte else 0.7)
    yy, xx = np.mgrid[0:h, 0:w].astype(np.float32)
    r2 = ((xx - w / 2) / (w / 2)) ** 2 + ((yy - h / 2) / (h / 2)) ** 2
    f = f * (1 - 0.12 * r2 / 2)
    ruido = rng.standard_normal(f.shape).astype(np.float32) * np.sqrt(np.clip(f, 0, 255) * 0.05 + 3.0 ** 2)
    return np.clip(f + ruido, 0, 255).astype(np.uint8)

def preparar(img, gsd_foto, seed=0):
    """Devuelve la imagen como la vería el OV9281 (gris, ruido, GSD objetivo) y su GSD (mm/px)."""
    s = gsd_foto / GSD_OBJETIVO_MM
    if s < 1:
        img = cv2.resize(img, None, fx=s, fy=s, interpolation=cv2.INTER_AREA)
    else:
        print(f"  Aviso: la foto ({gsd_foto} mm/px) ya tiene menos detalle que la cámara simulada; no se agranda.")
    gris = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY).astype(np.float32)
    gris = degradar(gris, np.random.default_rng(seed))
    return cv2.cvtColor(gris, cv2.COLOR_GRAY2BGR), max(gsd_foto, GSD_OBJETIVO_MM)

def posiciones(largo, tile):
    if largo <= tile:
        return [0]
    paso = int(tile * (1 - SOLAPE))
    return sorted(set(list(range(0, largo - tile, paso)) + [largo - tile]))

def detectar(img, tile, conf=CONF):
    """Corta en tiles del tamaño de entrenamiento, infiere cada uno y une los resultados con NMS."""
    tw, th = tile
    H, W = img.shape[:2]
    relleno = int(img.mean())
    cajas, confs, clases = [], [], []
    n, t0 = 0, time.perf_counter()
    for y0 in posiciones(H, th):
        for x0 in posiciones(W, tw):
            t = np.full((th, tw, 3), relleno, np.uint8)       # tile siempre completo: el modelo no lo agranda
            parte = img[y0:y0 + th, x0:x0 + tw]
            t[:parte.shape[0], :parte.shape[1]] = parte
            r = modelo.predict(t, imgsz=640, conf=conf, verbose=False)[0]
            n += 1
            for (x1, y1, x2, y2), c, k in zip(r.boxes.xyxy.cpu().numpy(), r.boxes.conf.cpu().numpy(),
                                              r.boxes.cls.cpu().numpy().astype(int)):
                cajas.append([float(x1 + x0), float(y1 + y0), float(x2 - x1), float(y2 - y1)])
                confs.append(float(c)); clases.append(int(k))
    dt = time.perf_counter() - t0
    finales = []
    for k in sorted(set(clases)):                            # NMS entre tiles (un daño en el solape cuenta una vez)
        idx = [i for i, c in enumerate(clases) if c == k]
        sel = cv2.dnn.NMSBoxes([cajas[i] for i in idx], [confs[i] for i in idx], conf, 0.5)
        finales += [(k, confs[idx[j]], cajas[idx[j]]) for j in np.array(sel).flatten()]
    return sorted(finales, key=lambda d: -d[1]), n, dt

COLORES = {"damage": (0, 0, 255), "dirt": (0, 200, 255)}

def dibujar(img, dets):
    out = img.copy()
    g = max(2, img.shape[1] // 700)
    for k, c, (x, y, w, h) in dets:
        nombre = NOMBRES[k]
        col = COLORES.get(nombre, (255, 0, 0))
        cv2.rectangle(out, (int(x), int(y)), (int(x + w), int(y + h)), col, g)
        cv2.putText(out, f"{nombre} {c:.2f}", (int(x), max(int(y) - 4 * g, 10)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.35 * g, col, g)
    return out

def procesar(ruta_foto, mostrar=True):
    img = leer(ruta_foto)
    gsd = GSD_FOTO_MM.get(ruta_foto.stem, GSD_POR_DEFECTO)
    vista, gsd_ef = preparar(img, gsd)
    dets, n, dt = detectar(vista, TILE)
    res = dibujar(vista, dets)
    SALIDA.mkdir(exist_ok=True)
    guardar(SALIDA / f"{ruta_foto.stem}.jpg", res)
    titulo = (f"{ruta_foto.name} | {gsd_ef:g} mm/px | {vista.shape[1]}x{vista.shape[0]} px | "
              f"{n} tiles en {dt:.1f} s | {sum(NOMBRES[d[0]] == 'damage' for d in dets)} daños")
    if mostrar:
        plt.figure(figsize=(16, 9))
        plt.imshow(cv2.cvtColor(res, cv2.COLOR_BGR2RGB)); plt.title(titulo); plt.axis("off"); plt.show()
    print(titulo)
    for k, c, (x, y, w, h) in dets:
        print(f"   {NOMBRES[k]:7s} conf {c:.2f}  {w:4.0f}x{h:<4.0f} px  ~{max(w, h) * gsd_ef / 10:5.1f} cm  "
              f"en ({x:.0f}, {y:.0f})")
    return res, dets

In [ ]:
# ============================================================
# 3. Todas las fotos de prueba
# ============================================================
fotos = sorted({p.resolve() for p in FOTOS.iterdir() if p.suffix.lower() in {".jpg", ".jpeg", ".png"}})
print(f"{len(fotos)} fotos en {FOTOS.name}\n")
for f in fotos:
    procesar(f)
print(f"\nImágenes guardadas en: {SALIDA}")